# 02 - Algorithm selection (P6.4/P6.5)

PPO / SAC / TD3 x FC / guidance x seeds 0,1,2 = 18 runs. The decision rule is fixed in `docs/algorithm_selection.md` section 6 *before* these runs. Cells are resumable: rerun after a disconnect.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)


In [ ]:
from training.algos import load_training_config
from training.runner import train
cfg = load_training_config()
BUDGET = int(cfg['budgets']['algorithm_selection_steps'])
print('budget per run:', BUDGET)

In [ ]:
TASKS = {
  'fc': {'task': 'fc', 'fc': {'domain_randomization': True, 'wind': True}},
  'guidance': {'task': 'guidance', 'guidance': {
      'platform': 'quadcopter', 'mode': 'apn_rl', 'target_set': 'evasive',
      'N_nav': 5.0}},
}
for algo in ('ppo', 'sac', 'td3'):
    for task, extra in TASKS.items():
        for seed in (0, 1, 2):
            run_id = f'p6_{algo}_{task}_s{seed}'
            summary = train(run_id, {'algo': algo, 'seed': seed,
                'total_steps': BUDGET, 'checkpoint_every': 10000, **extra})
            print(run_id, summary['num_timesteps'], 'done', summary['already_done'])

In [ ]:
import json
import pandas as pd
rows, curves = [], {}
for algo in ('ppo', 'sac', 'td3'):
    for task in TASKS:
        for seed in (0, 1, 2):
            run_id = f'p6_{algo}_{task}_s{seed}'
            d = ROOT / 'runs' / run_id
            done = json.loads((d / 'DONE').read_text())
            c = pd.read_csv(d / 'curves.csv')
            c['run'] = run_id
            curves[run_id] = c
            rows.append({'algo': algo, 'task': task, 'seed': seed,
                'steps': done['num_timesteps'],
                'final_reward': float(c['mean_reward'].iloc[-1]) if len(c) else float('nan'),
                'auc': float(c['mean_reward'].mean()) if len(c) else float('nan')})
table = pd.DataFrame(rows)
table

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, task in zip(axes, TASKS):
    for algo in ('ppo', 'sac', 'td3'):
        for seed in (0, 1, 2):
            c = curves[f'p6_{algo}_{task}_s{seed}']
            ax.plot(c['step'], c['mean_reward'], alpha=.45, label=f'{algo} {seed}')
    ax.set(title=task, xlabel='environment steps', ylabel='episode reward')
    ax.grid(alpha=.3)
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'figures' / 'algorithm_selection.png', dpi=150)

In [ ]:
summary = table.groupby(['task', 'algo']).agg(
    reward_mean=('final_reward', 'mean'), reward_min=('final_reward', 'min'),
    reward_max=('final_reward', 'max'), auc=('auc', 'mean')).reset_index()
summary['spread'] = summary.reward_max - summary.reward_min
summary.sort_values(['task', 'reward_mean'], ascending=[True, False])

## Decision

Apply the rule in `docs/algorithm_selection.md`: PPO stays unless a challenger has non-overlapping 95% intervals. Record the outcome in section 7 and set `selected_algo` in `configs/training.yaml`.